# Stg Layer — restaurant_mentions (mọi trường AI về quán ăn trích từ video)

1 row / **video × quán** (`restaurant_name_normalized`). Từ 2026-11-06 (datle) bảng này chứa **tất cả** các trường mà AI / tra cứu tạo ra cho một quán được nhắc trong video:

| Cột | Notebook ghi | Cách lấy |
| --- | --- | --- |
| tên, `address`, `ward`, `district`, `city`, `opening_hours`, `dishes` (`name`, `price_vnd`, `is_signature`), `mention_source`, `district_norm` | notebook này | `ai_query` trên **title + description** (`EXTRACTION_VERSION` 2), JSON thô cache ở `stg_restaurant_extraction_log` |
| `match_status`, `place_id`, `place_name`, `formatted_address`, `geo_ward`, `geo_city`, `latitude`, `longitude`, `place_types`, `name_similarity`, `geo_query_text`, `geocoded_at` | `stg/restaurant_geocode` | Photon (OpenStreetMap) + khớp với place Google mà auto_search đã tìm; tra 1 lần / khóa quán (tên + quận), chép vào mọi row của khóa đó (trước 06/11 là bảng riêng `stg_restaurant_geocode`) |
| `reviewer_rating`, `reviewer_summary`, `atmosphere`, `parking`, `service`, `price_range`, `would_return`, `video_address`, `review_model`, `review_video_seconds`, `reviewed_at` | `stg/restaurant_video_reviews` | Gemini xem video, điền các trường cho từng quán của video (thay transcript từ 06/11) |

**Incremental Flow (notebook này):**
1. `source_checksum = md5(title || description)` của latest snapshot mỗi video. Từ 06/11 không còn transcript: checksum của video từng trích kèm transcript được viết lại 1 lần (bước migration) để không bị trích lại vì lý do đó.
2. Chỉ gọi `ai_query` cho video **mới**, title / description **thay đổi**, hoặc trích bằng `EXTRACTION_VERSION` cũ và có quán; chỉ video `restaurant_review` (`stg_video_classification`).
3. MERGE JSON thô vào `stg_restaurant_extraction_log` (1 row / video, kể cả video không có quán).
4. MERGE (full sync) log → `stg_restaurant_mentions`. Cột geocode (theo khóa quán) và cột Gemini (theo video × quán) được **giữ lại** từ bảng hiện tại, nên trích lại hay video mới của quán đã biết không làm mất chúng; `address` lấy địa chỉ Gemini nghe trong video (`video_address`) khi description không có.
5. Verify.

**Notes:**
- `mention_source` tính bằng SQL (tên có trong title / description không), không dựa vào LLM: `title` / `both` / `description`.
- `restaurant_name_normalized` = lower, bỏ dấu, 1 space: khóa dedup. `district_norm` = cùng luật với Gold (`quan 3`, `binh thanh`, ...).
- Quán chỉ được nói trong lời (không có trong title / description) không còn được trích từ 06/11 (trade-off khi bỏ transcript).


In [ ]:
# --- 2026-11-04 (Free Edition daily DBU cap): LLM / Gemini steps run only in the daily 01h run ---
# Job parameter run_llm: "false" in the 3-hourly refresh runs, "true" when project1_auto_search triggers the job.
dbutils.widgets.text("run_llm", "true")
if dbutils.widgets.get("run_llm").strip().lower() != "true":
    dbutils.notebook.exit("skipped: run_llm = false (LLM steps run once a day, 01h)")


In [ ]:
%run ../gold/_udfs


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

VIDEOS_TABLE = f"{SILVER}.stg_videos"
LOG_TABLE = f"{SILVER}.stg_restaurant_extraction_log"
MENTIONS_TABLE = f"{SILVER}.stg_restaurant_mentions"
MODEL = "databricks-qwen3-next-80b-a3b-instruct"  # good Vietnamese, supports responseFormat
OLD_TRANSCRIPTS_TABLE = f"{SILVER}.stg_video_transcripts"  # retired 2026-11-06: only read once, to migrate checksums
OLD_GEOCODE_TABLE = f"{SILVER}.stg_restaurant_geocode"     # retired 2026-11-06: only read once, to copy lookups here
MAX_DESCRIPTION_CHARS = 6000
MAX_VIDEOS_PER_RUN = 500  # 2026-11-04: once a day (run_llm), was 300 per 3-hourly run; guard for the Free Edition daily usage quota (ai_query is not billed there); review videos, then new videos first, the rest wait for the next run

PROMPT = (
    "You extract restaurants (quán ăn / nhà hàng / tiệm) from a Vietnamese YouTube food video "
    "(its title and description). "
    "Only list specific places that the video visits or recommends. "
    "A numbered or listed entry that has its own address counts as a place even when its name is a dish "
    "(e.g. \"BÚN THỊT NƯỚNG MẮM CỦ KIỆU - ĐỊA CHỈ: 1 Nguyễn Trung Trực, Quận 1\"). "
    "Ignore the channel name, sponsors, and dishes that are not tied to a specific place. "
    "Copy addresses as written; city = Hồ Chí Minh when the text says Sài Gòn / SG / HCM. "
    "Use null for any address part the text does not state. "
    "For each place also list the dishes the video names or shows there (dishes), with price_vnd = the price "
    "in VND when it is stated (90.000đ / 90k / 90 nghìn -> 90000; a range or several sizes -> one dish entry per size), "
    "else null; is_signature = true for the dish the video presents as the place's specialty. "
    "opening_hours = the opening hours as stated (e.g. \"6:30 - 19:00\"), else null. Do not guess prices or hours."
)
EXTRACTION_VERSION = 2  # 2026-11-02: + dishes / price_vnd / is_signature / opening_hours (videos with restaurants are re-extracted once)
RESPONSE_FORMAT = """{"type":"json_schema","json_schema":{"name":"restaurants","schema":{"type":"object","properties":{"restaurants":{"type":"array","items":{"type":"object","properties":{"name":{"type":"string"},"address":{"type":["string","null"]},"ward":{"type":["string","null"]},"district":{"type":["string","null"]},"city":{"type":["string","null"]},"opening_hours":{"type":["string","null"]},"dishes":{"type":"array","items":{"type":"object","properties":{"name":{"type":"string"},"price_vnd":{"type":["integer","null"]},"is_signature":{"type":"boolean"}},"required":["name","price_vnd","is_signature"]}}},"required":["name","address","ward","district","city","opening_hours","dishes"]}}},"required":["restaurants"]},"strict":true}}"""
RESULT_SCHEMA = ("STRUCT<restaurants: ARRAY<STRUCT<name: STRING, address: STRING, ward: STRING, district: STRING, city: STRING, "
                 "opening_hours: STRING, dishes: ARRAY<STRUCT<name: STRING, price_vnd: BIGINT, is_signature: BOOLEAN>>>>>")

# Video classification is optional too: not classified = extract as before
CLASS_TABLE = f"{SILVER}.stg_video_classification"
if spark.catalog.tableExists(CLASS_TABLE):
    CLASS_SQL = f"SELECT video_id, is_food_review, video_type FROM {CLASS_TABLE}"
else:
    CLASS_SQL = ("SELECT CAST(NULL AS STRING) AS video_id, CAST(NULL AS BOOLEAN) AS is_food_review, "
                 "CAST(NULL AS STRING) AS video_type WHERE false")

print(f"Videos source: {VIDEOS_TABLE}")
print(f"Extraction log: {LOG_TABLE}")
print(f"Delta target: {MENTIONS_TABLE}")

# district_norm: same rule as Gold (Q.3 / quận 03 -> quan 3; TP Thủ Đức -> thu duc)
DISTRICT_NORM_SQL = r"""nullif(trim(regexp_replace(regexp_replace(regexp_replace(
    lower(unaccent_vi(coalesce({col}, ''))),
    '^(q\\.?\\s*|quan\\s+)0*(\\d+)$', 'quan $2'),
    '^(tp\\.?|thanh pho)\\s+', ''),
    '\\s+', ' ')), '')"""

# Columns kept from the current table on every full sync (written by later notebooks)
GEO_COLS = {"match_status": "STRING", "place_id": "STRING", "place_name": "STRING", "formatted_address": "STRING",
            "geo_ward": "STRING", "geo_city": "STRING", "latitude": "DOUBLE", "longitude": "DOUBLE",
            "place_types": "ARRAY<STRING>", "name_similarity": "DOUBLE", "geo_query_text": "STRING",
            "geocoded_at": "TIMESTAMP"}
REVIEW_COLS = {"reviewer_rating": "DOUBLE", "reviewer_summary": "STRING", "atmosphere": "STRING", "parking": "STRING",
               "service": "STRING", "price_range": "STRING", "would_return": "BOOLEAN", "video_address": "STRING",
               "review_model": "STRING", "review_video_seconds": "INT", "reviewed_at": "TIMESTAMP"}


In [ ]:
# --- Create tables if not exists ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
        video_id         STRING,
        source_checksum  STRING,
        raw_response     STRING,
        restaurant_count INT,
        model_name       STRING,
        extracted_at     TIMESTAMP
    )
    USING DELTA
    COMMENT 'Latest LLM restaurant extraction per video (1 row per video, also when it found no restaurant).'
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {MENTIONS_TABLE} (
        video_id                   STRING,
        restaurant_name            STRING,
        restaurant_name_normalized STRING,
        address                    STRING,
        ward                       STRING,
        district                   STRING,
        city                       STRING,
        mention_source             STRING,
        opening_hours              STRING,
        dishes                     ARRAY<STRUCT<name: STRING, price_vnd: BIGINT, is_signature: BOOLEAN>>,
        extraction_version         INT,
        source_checksum            STRING,
        extracted_at               TIMESTAMP,
        etl_date                   TIMESTAMP,
        district_norm              STRING,
        {", ".join(f"{c} {t}" for c, t in {**GEO_COLS, **REVIEW_COLS}.items())}
    )
    USING DELTA
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
    COMMENT 'Restaurants named per video with every AI field (grain: video_id x restaurant_name_normalized): ai_query extraction, geocode (per name + district) and Gemini video review.'
""")
# Tables created before EXTRACTION_VERSION 2: add the new columns
for table, cols in [(LOG_TABLE, {"extraction_version": "INT"}),
                    (MENTIONS_TABLE, {"opening_hours": "STRING",
                                      "dishes": "ARRAY<STRUCT<name: STRING, price_vnd: BIGINT, is_signature: BOOLEAN>>",
                                      "extraction_version": "INT", "district_norm": "STRING",
                                      **GEO_COLS, **REVIEW_COLS})]:
    missing = [c for c in cols if c not in spark.table(table).columns]
    if missing:
        spark.sql(f"ALTER TABLE {table} ADD COLUMNS ({', '.join(f'{c} {cols[c]}' for c in missing)})")
        print(f"➕ {table}: added {missing}")
print("Tables ready ✅")

# --- One-time migration (2026-11-06) -------------------------------------------------------------------------
# 1. Lookups of the retired stg_restaurant_geocode -> geocode columns here (no Photon call is repeated)
spark.sql(f"UPDATE {MENTIONS_TABLE} SET district_norm = {DISTRICT_NORM_SQL.format(col='district')} WHERE district_norm IS NULL AND district IS NOT NULL")
if spark.catalog.tableExists(OLD_GEOCODE_TABLE) and spark.sql(f"SELECT count(*) AS n FROM {MENTIONS_TABLE} WHERE match_status IS NOT NULL").first()["n"] == 0:
    spark.sql(f"""
        MERGE INTO {MENTIONS_TABLE} AS t
        USING {OLD_GEOCODE_TABLE} AS g
        ON t.restaurant_name_normalized = g.restaurant_name_normalized AND t.district_norm <=> g.district_norm
        WHEN MATCHED THEN UPDATE SET
            t.match_status = g.match_status, t.geo_query_text = g.query_text, t.geocoded_at = g.geocoded_at,
            t.name_similarity = g.name_similarity,
            -- rejected candidates (not_food / name_mismatch / no_result): status only
            t.place_id = CASE WHEN g.match_status IN ('matched', 'places_match', 'osm_approx') THEN g.place_id END,
            t.place_name = CASE WHEN g.match_status IN ('matched', 'places_match', 'osm_approx') THEN g.place_name END,
            t.formatted_address = CASE WHEN g.match_status IN ('matched', 'places_match', 'osm_approx') THEN g.formatted_address END,
            t.geo_ward = CASE WHEN g.match_status IN ('matched', 'places_match', 'osm_approx') THEN g.ward END,
            t.geo_city = CASE WHEN g.match_status IN ('matched', 'places_match', 'osm_approx') THEN g.city END,
            t.latitude = CASE WHEN g.match_status IN ('matched', 'places_match', 'osm_approx') THEN g.latitude END,
            t.longitude = CASE WHEN g.match_status IN ('matched', 'places_match', 'osm_approx') THEN g.longitude END,
            t.place_types = CASE WHEN g.match_status IN ('matched', 'places_match', 'osm_approx') THEN g.place_types END
    """)
    print(f"➕ copied the lookups of {OLD_GEOCODE_TABLE} into {MENTIONS_TABLE}")
# 2. Videos extracted with a transcript: checksum without it (so they are not re-extracted for that reason)
if spark.catalog.tableExists(OLD_TRANSCRIPTS_TABLE):
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW checksum_fix AS
        WITH v AS (
            SELECT video_id, title, description,
                   ROW_NUMBER() OVER (PARTITION BY video_id ORDER BY snapshot_date DESC, updated_at DESC) AS rn
            FROM {VIDEOS_TABLE}
        )
        SELECT v.video_id,
               md5(concat(concat_ws('||', coalesce(v.title, ''), coalesce(v.description, '')), '||', tr.transcript)) AS old_checksum,
               md5(concat_ws('||', coalesce(v.title, ''), coalesce(v.description, ''))) AS new_checksum
        FROM v JOIN {OLD_TRANSCRIPTS_TABLE} tr ON tr.video_id = v.video_id
        WHERE v.rn = 1 AND nullif(trim(tr.transcript), '') IS NOT NULL AND tr.transcript <> '[không có lời nói]'
    """)
    for table in (LOG_TABLE, MENTIONS_TABLE):
        spark.sql(f"""
            MERGE INTO {table} AS t USING checksum_fix AS s
            ON t.video_id = s.video_id AND t.source_checksum = s.old_checksum
            WHEN MATCHED THEN UPDATE SET t.source_checksum = s.new_checksum
        """)
    print(f"🔁 checksums without transcript: {spark.table('checksum_fix').count()} video(s) with a transcript")


In [ ]:
# --- Videos to (re-)extract: new, title / description changed, or extracted by an older
#     EXTRACTION_VERSION and it had restaurants (to get dishes / prices / hours) ---
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW videos_to_extract AS
    WITH latest AS (
        SELECT v.video_id, v.title, v.description, v.view_count,
               md5(concat_ws('||', coalesce(v.title, ''), coalesce(v.description, ''))) AS source_checksum,
               ROW_NUMBER() OVER (PARTITION BY v.video_id ORDER BY v.snapshot_date DESC, v.updated_at DESC) AS rn
        FROM {VIDEOS_TABLE} v
    )
    SELECT l.video_id, l.title, l.description, l.source_checksum
    FROM latest l
    LEFT JOIN {LOG_TABLE} g ON g.video_id = l.video_id
    LEFT JOIN ({CLASS_SQL}) vc ON vc.video_id = l.video_id
    WHERE l.rn = 1 AND (g.video_id IS NULL OR g.source_checksum <> l.source_checksum
                        OR (g.restaurant_count > 0 AND coalesce(g.extraction_version, 1) < {EXTRACTION_VERSION}))
      AND vc.video_type = 'restaurant_review'  -- 2026-11-04 (datle): only restaurant reviews (food_other rarely names a place; saves ai_query)
    -- reviews first, then never extracted, then the most viewed (2026-11-03: trending reviews get their restaurants first)
    ORDER BY coalesce(vc.video_type = 'restaurant_review', false) DESC, (g.video_id IS NULL) DESC, l.view_count DESC NULLS LAST, l.video_id
    LIMIT {MAX_VIDEOS_PER_RUN}
""")
todo_count = spark.table("videos_to_extract").count()
print(f"Videos to extract this run: {todo_count}")


In [ ]:
# --- Extract with ai_query and MERGE the raw result into the log (LLM is called once per video) ---
if todo_count > 0:
    prompt_sql = PROMPT.replace("'", "''")
    spark.sql(f"""
        MERGE INTO {LOG_TABLE} AS t
        USING (
            SELECT video_id, source_checksum, raw_response,
                   CAST(size(from_json(raw_response, '{RESULT_SCHEMA}').restaurants) AS INT) AS restaurant_count,
                   '{MODEL}' AS model_name, current_timestamp() AS extracted_at,
                   {EXTRACTION_VERSION} AS extraction_version
            FROM (
                SELECT video_id, source_checksum,
                       ai_query('{MODEL}',
                                concat('{prompt_sql}', '\\n\\nTITLE: ', coalesce(title, ''),
                                       '\\n\\nDESCRIPTION: ', left(coalesce(description, ''), {MAX_DESCRIPTION_CHARS})),
                                responseFormat => '{RESPONSE_FORMAT}') AS raw_response
                FROM videos_to_extract
            )
        ) AS s
        ON t.video_id = s.video_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✅ Extracted {todo_count} video(s) with {MODEL}")
else:
    print("Nothing new to extract ✅")


In [ ]:
# --- Full-sync MERGE log → mentions (atomic; deletes mentions that a re-extraction dropped) ---
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW mentions_src AS
    WITH exploded AS (
        SELECT g.video_id, g.source_checksum, g.extracted_at, coalesce(g.extraction_version, 1) AS extraction_version, r.*
        FROM {LOG_TABLE} g
        LATERAL VIEW explode(from_json(g.raw_response, '{RESULT_SCHEMA}').restaurants) x AS r
    ),
    cleaned AS (
        SELECT e.video_id, trim(e.name) AS restaurant_name,
               regexp_replace(lower(trim(unaccent_vi(normalize_keyword(e.name)))), '[^a-z0-9 ]', '') AS restaurant_name_normalized,
               nullif(trim(e.address), '') AS address, nullif(trim(e.ward), '') AS ward,
               nullif(trim(e.district), '') AS district, nullif(trim(e.city), '') AS city,
               nullif(trim(e.opening_hours), '') AS opening_hours,
               filter(e.dishes, d -> length(trim(d.name)) > 1) AS dishes, e.extraction_version,
               e.source_checksum, e.extracted_at,
               lower(unaccent_vi(coalesce(v.title, ''))) AS title_u,
               lower(unaccent_vi(coalesce(v.description, ''))) AS description_u
        FROM exploded e
        LEFT JOIN (
            SELECT video_id, title, description,
                   ROW_NUMBER() OVER (PARTITION BY video_id ORDER BY snapshot_date DESC, updated_at DESC) AS rn
            FROM {VIDEOS_TABLE}
        ) v ON v.video_id = e.video_id AND v.rn = 1
        WHERE length(trim(e.name)) > 1
    ),
    tagged AS (
        SELECT *,
            CASE
                WHEN instr(title_u, lower(unaccent_vi(restaurant_name))) > 0
                 AND instr(description_u, lower(unaccent_vi(restaurant_name))) > 0 THEN 'both'
                WHEN instr(title_u, lower(unaccent_vi(restaurant_name))) > 0 THEN 'title'
                ELSE 'description'
            END AS mention_source,
            ROW_NUMBER() OVER (PARTITION BY video_id, restaurant_name_normalized ORDER BY address IS NULL, restaurant_name) AS rn
        FROM cleaned
        WHERE restaurant_name_normalized <> ''
    ),
    src AS (SELECT *, {DISTRICT_NORM_SQL.format(col='district')} AS district_norm FROM tagged WHERE rn = 1),
    -- Gemini columns: per video x restaurant, kept from the current table
    reviewed AS (SELECT video_id, restaurant_name_normalized, {", ".join(REVIEW_COLS)} FROM {MENTIONS_TABLE}
                 WHERE reviewed_at IS NOT NULL),
    -- geocode columns: per restaurant key (name + district), kept from any current row of that key
    geo AS (
        SELECT restaurant_name_normalized, district_norm,
               max_by(struct({", ".join(GEO_COLS)}), geocoded_at) AS g
        FROM {MENTIONS_TABLE} WHERE match_status IS NOT NULL
        GROUP BY restaurant_name_normalized, district_norm
    )
    SELECT s.video_id, s.restaurant_name, s.restaurant_name_normalized,
           coalesce(s.address, r.video_address) AS address, s.ward, s.district, s.city,
           s.mention_source, s.opening_hours, s.dishes, s.extraction_version,
           s.source_checksum, s.extracted_at, current_timestamp() AS etl_date, s.district_norm,
           {", ".join(f"g.g.{c} AS {c}" for c in GEO_COLS)},
           {", ".join(f"r.{c}" for c in REVIEW_COLS)}
    FROM src s
    LEFT JOIN reviewed r ON r.video_id = s.video_id AND r.restaurant_name_normalized = s.restaurant_name_normalized
    LEFT JOIN geo g ON g.restaurant_name_normalized = s.restaurant_name_normalized AND g.district_norm <=> s.district_norm
""")

spark.table("mentions_src").select(*spark.table(MENTIONS_TABLE).columns).createOrReplaceTempView("mentions_src_ordered")
spark.sql(f"""
    MERGE INTO {MENTIONS_TABLE} AS t
    USING mentions_src_ordered AS s
    ON t.video_id = s.video_id AND t.restaurant_name_normalized = s.restaurant_name_normalized
    WHEN MATCHED AND (t.source_checksum <> s.source_checksum
                      OR NOT (t.extraction_version <=> s.extraction_version)
                      OR t.district_norm IS NULL AND s.district_norm IS NOT NULL) THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
print(f"✅ Merged into {MENTIONS_TABLE}")


In [ ]:
# --- Verify ---
stats = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT video_id, restaurant_name_normalized) AS n_grain,
           COUNT(DISTINCT video_id) AS videos, COUNT(address) AS with_address,
           COUNT_IF(size(dishes) > 0) AS with_dishes, COUNT_IF(exists(dishes, d -> d.price_vnd IS NOT NULL)) AS with_price,
           COUNT(opening_hours) AS with_hours, COUNT_IF(match_status IN ('matched', 'places_match', 'osm_approx')) AS located,
           COUNT(reviewed_at) AS reviewed
    FROM {MENTIONS_TABLE}
""").collect()[0]
log = spark.sql(f"SELECT COUNT(*) AS n, COUNT_IF(restaurant_count = 0) AS empty FROM {LOG_TABLE}").collect()[0]
print(f"📊 Extraction log: {log['n']} videos ({log['empty']} without any restaurant)")
print(f"📊 {MENTIONS_TABLE}: {stats['n']} mentions in {stats['videos']} videos, {stats['with_address']} with an address, "
      f"{stats['with_dishes']} with dishes, {stats['with_price']} with a price, {stats['with_hours']} with opening hours, "
      f"{stats['located']} located (geocode), {stats['reviewed']} reviewed by Gemini")
print(f"📊 Grain check (video_id × restaurant_name_normalized): {stats['n']} rows vs {stats['n_grain']} distinct "
      + ("✅" if stats["n"] == stats["n_grain"] else "❌ duplicates"))
print("\n--- stg_restaurant_mentions ETL complete ---")
